In [ ]:
import sys
sys.path.append("src")

from data_loader import load_all_documents

docs = load_all_documents("data/reports")
print(f"Loaded {len(docs)} documents.")

In [ ]:
# Inspect the first loaded PDF page
if docs:
    print(docs[0].metadata)
    print(docs[0].page_content[:500])

# RAG Grounded Answer Generation

Runs the same path as the Streamlit *Search Medical Reports* page (`AnswerService`) directly in Jupyter - no `app.py`, no Streamlit.
Run the cells **in order**. Nothing here re-indexes, re-embeds, uploads or deletes data (read-only on `chroma_store/`).
Use synthetic patient records only.

In [1]:
# Cell 1 - Imports and environment configuration
import os, sys, logging
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():           # notebook started from a sub-folder
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")                # must run BEFORE importing src.config

from IPython.display import Markdown, display
from src import config
from src.vector_store import ChromaVectorStore
from src.answer_service import AnswerService
from src.llm import build_llm

logging.getLogger("src").setLevel(logging.WARNING)

print("LLM provider      :", config.LLM_PROVIDER)
print("LLM model         :", config.LLM_MODEL if config.LLM_PROVIDER != "qwen_local" else config.QWEN_MODEL_NAME)
print("LLM base URL      :", config.LLM_BASE_URL or "-")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))   # value is never printed
print("Chroma directory  :", config.CHROMA_DIR)
print("Collection        :", config.COLLECTION_NAME)
print("Embedding model   :", config.EMBEDDING_MODEL)
print()
print("NOTE: src.config is read once at import time. After editing .env or model settings, "
      "RESTART the kernel and run the cells again.")

d:\Upgrad_RAG_Project\GeneralPhysicianRagFT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM provider      : openai
LLM model         : gpt-4.1
LLM base URL      : -
OPENAI_API_KEY set: True
Chroma directory  : D:\Upgrad_RAG_Project\GeneralPhysicianRagFT\chroma_store
Collection        : lab_reports
Embedding model   : all-MiniLM-L6-v2

NOTE: src.config is read once at import time. After editing .env or model settings, RESTART the kernel and run the cells again.


In [2]:
# Cell 2 - Open the EXISTING persistent ChromaDB collection (read-only use)
store = ChromaVectorStore(
    persist_dir=config.CHROMA_DIR,
    collection_name=config.COLLECTION_NAME,
    embedding_model=config.EMBEDDING_MODEL,
)
chunk_count = store.collection.count()
print(f"Collection '{config.COLLECTION_NAME}' in {config.CHROMA_DIR}: {chunk_count} chunks")
if chunk_count == 0:
    raise RuntimeError("The collection is empty. Index reports first (via the app's upload flow); "
                       "this notebook deliberately does not ingest anything.")
print("Indexed documents found - OK to continue.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3269.66it/s]


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Chroma collection 'lab_reports' at D:\Upgrad_RAG_Project\GeneralPhysicianRagFT\chroma_store (604 existing vectors)
Collection 'lab_reports' in D:\Upgrad_RAG_Project\GeneralPhysicianRagFT\chroma_store: 604 chunks
Indexed documents found - OK to continue.


In [3]:
# Cell 3 - Create the reusable AnswerService (same construction as app.py: AnswerService(store))
# The LLM is built lazily on first use via build_llm(), exactly as in the app.
answer_service = AnswerService(store)
print("AnswerService ready.")

AnswerService ready.


In [4]:
# Cell 4 - Ask a question (edit and re-run this cell)
question   = "What is the hemoglobin count for patient P0001?"
patient_id = None   # optional, e.g. "P0001" (the question may also contain an ID or a unique full name)
report_id  = None   # optional: restrict to one report; None = all reports

result = answer_service.answer_question(
    question,
    patient_id_field=patient_id,
    report_id=report_id,
)
print("status:", result["status"])

status: ok


In [5]:
# Cell 5 - Display the grounded answer (no chunks, vectors, distances or chunk ids)
def show_result(question, result):
    out = [f"**Question:**\n\n{question}", f"**Answer:**\n\n{result['answer']}"]
    if result["status"] == "ok" and result["sources"]:
        for s in result["sources"]:
            out.append(f"**Source:** {s['report_name']}  \n**Page:** {s['page']}  \n"
                       f"**Report Date:** {s['report_date'] or 'not recorded'}")
        out.append(f"**Answer Method:** {result['method'] or 'n/a'}")
    elif result["status"] != "ok":
        out.append(f"_Status: {result['status']}_")
    display(Markdown("\n\n".join(out)))

show_result(question, result)

**Question:**

What is the hemoglobin count for patient P0001?

**Answer:**

The hemoglobin count for patient P0001 is 13.6 g/dL, which is marked as NORMAL and within the reference range of 13–17 g/dL. This information is from the report dated 2026-11-20.

**Source:** R0201_P0001_Synthetic_Followup.pdf  
**Page:** 1  
**Report Date:** 2026-11-20

**Answer Method:** llm

In [6]:
# Cell 6 (optional) - Verify the configured model (Qwen / OpenAI) is really invoked
print("Provider:", config.LLM_PROVIDER)
if config.LLM_PROVIDER == "none":
    print("LLM_PROVIDER=none -> deterministic answers only; no model will be called.")
else:
    # (a) independent model-generation test
    llm_ok = False
    try:
        llm = build_llm()
        reply = llm.generate("You are a test assistant.", "Reply with the single word: OK")
        llm_ok = bool(reply and reply.strip())
        print("Model invoked successfully. Reply:", reply.strip()[:80])
    except Exception as exc:
        print(f"Model NOT available ({type(exc).__name__}): {str(exc)[:200]}")
        print("Check that the server is running (e.g. Ollama at LLM_BASE_URL) or that OPENAI_API_KEY is set.")

    # (b) RAG summary question through AnswerService
    q = "Summarize the latest medical report for patient P0001."
    r = answer_service.answer_question(q)
    show_result(q, r)
    if r["method"] == "llm":
        print("-> This answer was generated by the configured LLM (and passed numeric validation).")
    elif r["method"] == "deterministic":
        print("-> Deterministic fallback was used; the LLM was unavailable, failed, or its output failed validation."
              + ("" if llm_ok else " (The independent model test above also failed.)"))
    else:
        print("-> No LLM-generated answer was produced (status:", r["status"], ").")

Provider: openai
Model invoked successfully. Reply: OK


**Question:**

Summarize the latest medical report for patient P0001.

**Answer:**

The latest medical report for patient P0001 (dated 2026-11-20, report R0201) shows the following:

- Diabetes Panel: HbA1c (6.8%), fasting plasma glucose (142 mg/dL), and estimated average glucose (148 mg/dL) are all flagged as HIGH.
- Lipid Profile: LDL cholesterol is HIGH at 108 mg/dL. Total cholesterol, HDL cholesterol, and triglycerides are within normal ranges.
- Kidney Function: Serum creatinine and blood urea nitrogen (BUN) are within normal ranges.
- Complete Blood Count (CBC): All values, including hemoglobin, RBC count, hematocrit, MCV, MCH, MCHC, WBC count, and platelet count, are within normal ranges.

For interpretation of these results, please consult your doctor. This summary is based on report R0201_P0001_Synthetic_Followup.pdf dated 2026-11-20.

**Source:** R0201_P0001_Synthetic_Followup.pdf  
**Page:** 2  
**Report Date:** 2026-11-20

**Source:** R0201_P0001_Synthetic_Followup.pdf  
**Page:** 1  
**Report Date:** 2026-11-20

**Answer Method:** llm

-> This answer was generated by the configured LLM (and passed numeric validation).


In [8]:
# Cell 7 - Reusable helper
def ask_medical_report(question: str, patient_id: str | None = None, report_id: str | None = None):
    """Resolve patient identity, retrieve with patient filters, answer, and show answer + sources."""
    try:
        result = answer_service.answer_question(question, patient_id_field=patient_id, report_id=report_id)
    except Exception as exc:   # e.g. Chroma/embedding problems; never print config values
        display(Markdown(f"**Error:** could not answer ({type(exc).__name__}). Check Cells 1-3."))
        return None
    show_result(question, result)
    return result

# Examples (uncomment to run)
ask_medical_report("What is the hemoglobin count for patient P0001?")
# ask_medical_report("What is the HbA1c level?", patient_id="P0001")
# ask_medical_report("Summarize the latest report for P0001")
# ask_medical_report("Compare hemoglobin results across reports for P0001")

**Question:**

What is the hemoglobin count for patient P0001?

**Answer:**

The hemoglobin count for patient P0001 is 13.6 g/dL, which is marked as NORMAL and within the reference range of 13–17 g/dL. This information is from the report dated 2026-11-20.

**Source:** R0201_P0001_Synthetic_Followup.pdf  
**Page:** 1  
**Report Date:** 2026-11-20

**Answer Method:** llm

{'status': 'ok',
 'answer': 'The hemoglobin count for patient P0001 is 13.6 g/dL, which is marked as NORMAL and within the reference range of 13–17 g/dL. This information is from the report dated 2026-11-20.',
 'patient_id': 'P0001',
 'sources': [{'report_id': 'R0201',
   'report_name': 'R0201_P0001_Synthetic_Followup.pdf',
   'report_date': '2026-11-20',
   'page': 1}],
 'intent': 'lab_value',
 'method': 'llm'}